# LangGraph 006 — Sequential Workflows

The four steps — **state → nodes → edges → compile and invoke** — on three
workflows. Needs `langgraph`; **no API key** (the model calls are scripted).

| Part | What we check |
|---|---|
| A | the BMI graph, and every boundary it labels |
| B | returning the state instead of the update **duplicates** entries |
| C | prompt chaining keeps **3** values where a chain keeps **1** |
| D | printing the graph without installing anything |
| E | the homework: add a scoring node |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The BMI workflow

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. the state
class BMIState(TypedDict):
    weight_kg: float
    height_m: float
    bmi: float
    category: str

# 2. the nodes - plain Python functions
def calculate_bmi(state: BMIState) -> dict:
    return {"bmi": round(state["weight_kg"] / state["height_m"] ** 2, 2)}

def label_bmi(state: BMIState) -> dict:
    bmi = state["bmi"]
    if bmi < 18.5:
        label = "Underweight"
    elif bmi < 25:
        label = "Normal"
    elif bmi < 30:
        label = "Overweight"
    else:
        label = "Obese"
    return {"category": label}

# 3. the graph: nodes, edges, compile
graph = StateGraph(BMIState)
graph.add_node("calculate_bmi", calculate_bmi)
graph.add_node("label_bmi", label_bmi)
graph.add_edge(START, "calculate_bmi")
graph.add_edge("calculate_bmi", "label_bmi")
graph.add_edge("label_bmi", END)
workflow = graph.compile()

# 4. run it
print(workflow.invoke({"weight_kg": 80, "height_m": 1.73, "bmi": 0.0, "category": ""}))
# {'weight_kg': 80, 'height_m': 1.73, 'bmi': 26.73, 'category': 'Overweight'}

In [ ]:
out = workflow.invoke({"weight_kg": 80, "height_m": 1.73, "bmi": 0.0, "category": ""})
assert out["bmi"] == 26.73 and out["category"] == "Overweight"
assert out["weight_kg"] == 80            # the inputs are carried through

for w, expected in ((50, "Underweight"), (65, "Normal"), (80, "Overweight"), (95, "Obese")):
    r = workflow.invoke({"weight_kg": w, "height_m": 1.73, "bmi": 0.0, "category": ""})
    print(f"{w:>3} kg -> {r['bmi']:>6}  {r['category']}")
    assert r["category"] == expected

## Part B — Return the update, not the state

On a plain state both styles agree. Add a reducer and they stop agreeing.

In [ ]:
import operator
from typing import Annotated

class Reduced(TypedDict):
    log: Annotated[list, operator.add]

def mutating(state):             # mutate the state, then return the whole thing
    state["log"].append("x")
    return state

def partial(state):              # return only what changed
    return {"log": ["x"]}

for name, fn in (("mutate + return state", mutating), ("return the update", partial)):
    g = StateGraph(Reduced)
    g.add_node("n", fn); g.add_edge(START, "n"); g.add_edge("n", END)
    print(f"{name:<22} -> {g.compile().invoke({'log': ['start']})['log']}")
# mutate + return state  -> ['start', 'x', 'start', 'x']
# return the update      -> ['start', 'x']
#
# The mutating node adds 'x' to the list in place AND hands the whole list back,
# so the reducer appends all of it a second time. Return the update, not the state.

In [ ]:
def run(fn, start=("start",)):
    g = StateGraph(Reduced)
    g.add_node("n", fn); g.add_edge(START, "n"); g.add_edge("n", END)
    return g.compile().invoke({"log": list(start)})["log"]

assert run(mutating) == ["start", "x", "start", "x"]
assert run(partial) == ["start", "x"]
print("no exception either way - the mutating node just silently doubles the list")

## Part C — Prompt chaining

In [ ]:
# The same two-step chain with the model calls scripted, so it runs with no key.
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

def create_outline(state: BlogState) -> dict:
    return {"outline": f"1. why {state['title']} matters  2. evidence  3. outlook"}

def create_blog(state: BlogState) -> dict:
    return {"content": f"[900-word blog on '{state['title']}' following "
                       f"{len(state['outline'])} characters of outline]"}

graph = StateGraph(BlogState)
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

final = graph.compile().invoke({"title": "Rise of AI in India", "outline": "", "content": ""})
print(len(final), "values kept:", sorted(final))
print("outline:", final["outline"])
# 3 values kept: ['content', 'outline', 'title']
# outline: 1. why Rise of AI in India matters  2. evidence  3. outlook
#
# A LangChain chain of the same two steps returns the blog alone: the outline it
# worked from is gone. The state keeps everything the workflow produced.

In [ ]:
from langchain_core.runnables import RunnableLambda
chain = (RunnableLambda(lambda t: f"1. why {t} matters  2. evidence  3. outlook")
         | RunnableLambda(lambda o: f"[900-word blog following {len(o)} characters of outline]"))
chain_out = chain.invoke("Rise of AI in India")

assert len(final) == 3 and isinstance(chain_out, str)
print("graph keeps:", sorted(final))
print("chain keeps: the final string only ->", chain_out[:50])

## Part D — Seeing the graph

In [ ]:
app = graph.compile()
print([n for n in app.get_graph().nodes])
# ['__start__', 'create_outline', 'create_blog', '__end__']

print(app.get_graph().draw_mermaid()[:120])   # works with nothing extra installed
# ---
# config:
#   flowchart:
#     curve: linear
# ---
# graph TD;

# app.get_graph().print_ascii()
# ImportError: Install grandalf to draw graphs: `pip install grandalf`
# In a notebook the usual picture is:
#   from IPython.display import Image
#   Image(app.get_graph().draw_mermaid_png())   # needs network access

## Part E — The homework

Add a third node that scores the blog out of 10. Three things change: the state
gains a key, the node is added, and the edge that used to run to `END` now runs
to the new node.

In [ ]:
class ScoredBlog(TypedDict):
    title: str
    outline: str
    content: str
    score: int                      # 1. the new key

def evaluate(state: ScoredBlog) -> dict:
    return {"score": min(10, len(state["content"]) // 10)}   # scripted "model"

g = StateGraph(ScoredBlog)
g.add_node("create_outline", create_outline)
g.add_node("create_blog", create_blog)
g.add_node("evaluate", evaluate)                              # 2. the new node
g.add_edge(START, "create_outline")
g.add_edge("create_outline", "create_blog")
g.add_edge("create_blog", "evaluate")                         # 3. the rewired edge
g.add_edge("evaluate", END)

out = g.compile().invoke({"title": "Rise of AI in India", "outline": "", "content": "", "score": 0})
print({k: (v if k != "content" else v[:40] + "...") for k, v in out.items()})
assert set(out) == {"title", "outline", "content", "score"}
print("four values in the final state, one per step of the workflow")

## What to take away

- The pattern never changes: **state → StateGraph → add_node → add_edge →
  compile → invoke**.
- A node takes the state and returns **the keys it changed**.
- Under a reducer, returning the whole state duplicates entries — silently.
- The state keeps every intermediate, which a plain chain does not.

## Exercises

1. Make `label_bmi` return the whole state instead of `{"category": ...}`. Does
   the BMI example still work? Why does the duplicate not appear here?
2. Add a `height_cm` input and a node that converts it to metres before the BMI
   node runs.
3. Score the blog with a second scripted "model" that returns feedback as well as
   a number, and keep both in the state.